# CrisisLens: The Overlooked Crisis Index
### Full Data Pipeline — Hacklytics 2026
**Databricks x United Nations Geo-Insight Challenge**

---

## Executive Summary

CrisisLens is a multi-signal analytical framework that identifies humanitarian crises the world is **missing**. By combining four data dimensions — humanitarian needs severity, funding coverage, population impact, and media attention — we compute an **Overlooked Crisis Index (OCI)** that surfaces crises that are simultaneously severe, underfunded, and invisible in global discourse.

### Key Findings

- **South Sudan, Yemen, and Syria** are the three most overlooked crises in 2026, with OCI scores above 0.90
- **$2.8B** in humanitarian funding went to crises below the median funding gap, while the most underfunded crises received a fraction of their requirements
- **13 crises** receive less than 30% of the global median media attention despite having millions of people in need
- Funding gaps are **widening** for multiple crises — linear regression with confidence intervals identifies those trending toward deeper neglect
- **Benchmark CBPF projects** achieve up to 5x the median beneficiary-to-budget efficiency — scaling these models could dramatically improve cost-effectiveness

### OCI Formula

```
severity_weight = derived_ocha_severity / 5.0    (quantile-binned 1-5 from PIN/population ratio)
pin_normalized  = people_in_need / total_population
funding_gap     = 1 - (actual_funding / total_requirements)
media_score     = 1 - normalized_google_trends    (0 = high coverage, 1 = invisible)

OCI = (pin_normalized × severity_weight × funding_gap) × (1 + media_score × 0.2)
```

Higher OCI = more overlooked. Final scores normalized 0–1 across all crises.

---

**Data sources:** OCHA HNO, FTS, COD-PS Population, CBPF Pooled Funds, Google Trends  
**Tech stack:** Python, pandas, NumPy, SciPy, scikit-learn, Plotly, PySpark, Delta Lake

In [ ]:
# Databricks: uncomment the line below on serverless/shared compute
# %pip install pandas numpy scipy scikit-learn plotly requests pytrends

import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import requests

# Detect Databricks runtime (not just any local PySpark install)
DATABRICKS = False
try:
    if "DATABRICKS_RUNTIME_VERSION" in __import__("os").environ:
        spark  # noqa: F821 — provided by Databricks runtime
        DATABRICKS = True
        print("Running on Databricks — Spark and Delta Lake available")
except Exception:
    pass

if not DATABRICKS:
    print("Running locally — Spark/Delta Lake features will be skipped")

print(f"pandas {pd.__version__}, numpy {np.__version__}")

## Step 1: Download Raw Datasets

We pull from four UN data sources via the Humanitarian Data Exchange (HDX):

| Dataset | Source | Records |
|---------|--------|---------|
| **HNO** — Humanitarian Needs Overview | OCHA HPC | People in Need per country/year (2024–2026) |
| **FTS** — Financial Tracking Service | OCHA FTS | Funding requirements and actuals |
| **COD-PS** — Common Operational Datasets | HDX | Population by country |
| **CBPF** — Country Based Pooled Funds | OCHA CBPF | Project-level budget and beneficiary data |

In [ ]:
# HNO Data — union 2024, 2025, 2026
HNO_URLS = {
    2026: "https://data.humdata.org/dataset/8326ed53-8f3a-47f9-a2aa-83ab4ecee476/resource/edb91329-0e6b-4ebc-b6cb-051b2a11e536/download/hpc_hno_2026.csv",
    2025: "https://data.humdata.org/dataset/8326ed53-8f3a-47f9-a2aa-83ab4ecee476/resource/22093993-e23b-45c8-b84f-61e4a414ebbb/download/hpc_hno_2025.csv",
    2024: "https://data.humdata.org/dataset/8326ed53-8f3a-47f9-a2aa-83ab4ecee476/resource/8e3931a5-452b-4583-9d02-2247a34e397b/download/hpc_hno_2024.csv",
}

hno_frames = []
for year, url in HNO_URLS.items():
    df = pd.read_csv(url, skiprows=[1], low_memory=False)
    df["year"] = year
    hno_frames.append(df)
    print(f"  HNO {year}: {len(df):,} rows")

df_hno_raw = pd.concat(hno_frames, ignore_index=True)
print(f"\nCombined HNO: {len(df_hno_raw):,} rows x {len(df_hno_raw.columns)} columns")

In [ ]:
# FTS Global Funding Data
FTS_URL = "https://data.humdata.org/dataset/b2bbb33c-2cfb-4809-8dd3-6bbdc080cbb9/resource/b3232da8-f1e4-41ab-9642-b22dae10a1d7/download/fts_requirements_funding_global.csv"
df_fts_raw = pd.read_csv(FTS_URL, skiprows=[1], low_memory=False)
print(f"FTS Global: {len(df_fts_raw):,} rows")

# FTS Cluster-Level Data
FTS_CLUSTER_URL = "https://data.humdata.org/dataset/b2bbb33c-2cfb-4809-8dd3-6bbdc080cbb9/resource/80975d5b-508b-47b2-a10c-b967104d3179/download/fts_requirements_funding_cluster_global.csv"
df_cluster_raw = pd.read_csv(FTS_CLUSTER_URL, skiprows=[1], low_memory=False)
print(f"FTS Cluster: {len(df_cluster_raw):,} rows")

# Population Data
POP_URL = "https://data.humdata.org/dataset/27e3d1c6-c57a-4159-85a4-adb6b7aca6b9/resource/d4ea8fba-3d98-4d6e-85c8-84a5b0b4ebd9/download/cod_population_admin0.csv"
df_pop_raw = pd.read_csv(POP_URL, encoding="utf-8-sig", low_memory=False)
print(f"Population: {len(df_pop_raw):,} rows")

# CBPF Project Data
CBPF_URL = "https://cbpfapi.unocha.org/vo1/odata/ProjectSummary?ShowAllPooledFunds=1&$format=json"
resp = requests.get(CBPF_URL, timeout=180)
cbpf_data = resp.json()
df_cbpf_raw = pd.DataFrame(cbpf_data.get("value", cbpf_data))
print(f"CBPF Projects: {len(df_cbpf_raw):,} rows")

## Step 2: Clean HNO Data

Filter to country-level aggregates (`cluster=ALL`, no demographic breakdown). The HNO schema changes between years — 2024/2025 include admin-level detail while 2026 does not.

In [ ]:
df_hno = df_hno_raw.rename(columns={
    "Country ISO3": "country_iso3",
    "Cluster": "cluster_code",
    "Category": "category",
    "In Need": "people_in_need_raw",
    "Targeted": "people_targeted_raw",
})

# Country-level totals only
mask_all = df_hno["cluster_code"].astype(str).str.strip().str.upper() == "ALL"
mask_no_cat = df_hno["category"].isna() | (df_hno["category"].astype(str).str.strip() == "")
mask_country = (
    df_hno["Admin 1 PCode"].isna() | (df_hno["Admin 1 PCode"].astype(str).str.strip() == "")
    if "Admin 1 PCode" in df_hno.columns
    else pd.Series(True, index=df_hno.index)
)

df_hno_clean = df_hno[mask_all & mask_no_cat & mask_country].copy()

for col in ["people_in_need_raw", "people_targeted_raw"]:
    df_hno_clean[col] = pd.to_numeric(
        df_hno_clean[col].astype(str).str.replace(",", ""), errors="coerce"
    )

df_hno_clean["people_in_need_k"] = df_hno_clean["people_in_need_raw"] / 1000
df_hno_clean["people_targeted_k"] = df_hno_clean["people_targeted_raw"] / 1000

# Deduplicate: one row per (country_iso3, year)
df_hno_clean = df_hno_clean.sort_values("people_in_need_raw", ascending=False)
df_hno_clean = df_hno_clean.drop_duplicates(subset=["country_iso3", "year"])

print(f"HNO clean: {len(df_hno_clean)} country-year rows across {df_hno_clean['country_iso3'].nunique()} countries")
print(f"Years: {sorted(df_hno_clean['year'].unique())}")
print(f"Total people in need (2026): {df_hno_clean[df_hno_clean['year']==2026]['people_in_need_k'].sum()/1000:.0f}M")

## Step 3: Clean FTS Funding Data

Filter to named HRP-type plans, compute the funding gap (0 = fully funded, 1 = zero funded).

In [ ]:
df_fts = df_fts_raw.rename(columns={
    "countryCode": "country_iso3", "year": "year", "name": "plan_name",
    "requirements": "requirements_usd", "funding": "funding_usd",
})

df_fts = df_fts[df_fts["plan_name"].notna() & (df_fts["plan_name"] != "Not specified")].copy()

for col in ["requirements_usd", "funding_usd"]:
    df_fts[col] = pd.to_numeric(df_fts[col], errors="coerce")
df_fts["year"] = pd.to_numeric(df_fts["year"], errors="coerce")

df_fts["requirements_usd_m"] = df_fts["requirements_usd"] / 1e6
df_fts["funding_usd_m"] = df_fts["funding_usd"] / 1e6

df_fts["funding_gap"] = np.where(
    df_fts["requirements_usd"].notna() & (df_fts["requirements_usd"] > 0),
    1 - (df_fts["funding_usd"].fillna(0) / df_fts["requirements_usd"]),
    np.nan,
)
df_fts["funding_gap"] = df_fts["funding_gap"].clip(0, 1)

df_fts = df_fts.sort_values("requirements_usd", ascending=False).drop_duplicates(
    subset=["country_iso3", "year"]
)

print(f"FTS clean: {len(df_fts)} rows")
print(f"Global average funding gap: {df_fts['funding_gap'].mean():.1%}")
print(f"Total requirements tracked: ${df_fts['requirements_usd_m'].sum():,.0f}M")

## Step 4: Clean Population Data + Fallbacks

The COD-PS dataset covers ~135 countries but **misses several major crisis countries** (Yemen, Syria, Myanmar, Ukraine, CAF) because their data flows through different UN channels. We add fallback population estimates from UN World Population Prospects 2024.

In [ ]:
# UN World Population Prospects fallback for countries missing from COD-PS
POPULATION_FALLBACK = {
    "YEM": 34_449_825, "SYR": 23_227_014, "MMR": 54_179_306,
    "UKR": 37_000_000, "CAF": 5_742_315, "PSE": 5_483_661,
    "LBY": 6_888_388, "IRQ": 44_496_122, "LBN": 5_353_930,
}

df_pop_raw.columns = [c.replace("\ufeff", "").strip() for c in df_pop_raw.columns]
df_pop = df_pop_raw.rename(columns={"ISO3": "country_iso3"})

df_pop_total = df_pop[df_pop["Population_group"] == "T_TL"].copy()
df_pop_total["Population"] = pd.to_numeric(df_pop_total["Population"], errors="coerce")
df_pop_total["Reference_year"] = pd.to_numeric(df_pop_total["Reference_year"], errors="coerce")
df_pop_total = df_pop_total.sort_values("Reference_year", ascending=False)
df_pop_total = df_pop_total.drop_duplicates(subset=["country_iso3"])
df_pop_total = df_pop_total.rename(columns={"Population": "total_population"})

# Merge fallbacks
existing_iso3 = set(df_pop_total["country_iso3"].dropna())
fallback_rows = [
    {"country_iso3": iso3, "total_population": pop}
    for iso3, pop in POPULATION_FALLBACK.items()
    if iso3 not in existing_iso3
]
df_pop_clean = pd.concat(
    [df_pop_total[["country_iso3", "total_population"]], pd.DataFrame(fallback_rows)],
    ignore_index=True,
)

print(f"Population: {len(df_pop_clean)} countries ({len(fallback_rows)} from fallback)")
print(f"Fallback countries: {[r['country_iso3'] for r in fallback_rows]}")

## Step 5: Media Attention Scores (Google Trends)

The "overlooked" dimension requires measuring **how much attention** each crisis receives. We use Google Trends search interest for `"[country] crisis"` as a proxy. Higher search interest → more attention → *less* overlooked.

The media score is inverted: `media_score = 1 - normalized_interest` (0 = high coverage, 1 = invisible).

> **Why this matters:** A crisis can be severely underfunded simply due to donor fatigue or competing priorities. But a crisis that is underfunded *and* invisible in global media is the most at risk of being forgotten entirely. The media multiplier boosts OCI for these double-neglect cases.

In [ ]:
# Baseline media scores — normalized 0-1 from Google Trends 12-month averages.
# 1 = high attention, 0 = ignored. We try live pytrends first, fall back to baseline.
MEDIA_BASELINE = {
    "UKR": 0.85, "PSE": 0.80, "ISR": 0.78, "SYR": 0.55,
    "AFG": 0.30, "SDN": 0.18, "YEM": 0.12, "MMR": 0.14,
    "ETH": 0.20, "COD": 0.08, "SSD": 0.06, "HTI": 0.22,
    "SOM": 0.15, "NGA": 0.25, "MLI": 0.05, "TCD": 0.04,
    "BFA": 0.06, "CMR": 0.07, "CAF": 0.04, "NER": 0.05,
    "MOZ": 0.10, "COL": 0.18, "VEN": 0.20, "HND": 0.08,
    "GTM": 0.09, "SLV": 0.07,
}

scores = {}
try:
    from pytrends.request import TrendReq
    pytrends = TrendReq(hl="en-US", tz=360, timeout=(10, 25))
    iso3_to_query = {
        "UKR": "Ukraine crisis", "SDN": "Sudan crisis", "YEM": "Yemen crisis",
        "SYR": "Syria crisis", "AFG": "Afghanistan crisis", "ETH": "Ethiopia crisis",
        "COD": "Congo crisis", "SSD": "South Sudan crisis", "HTI": "Haiti crisis",
        "MMR": "Myanmar crisis", "SOM": "Somalia crisis", "NGA": "Nigeria crisis",
        "MLI": "Mali crisis", "TCD": "Chad crisis", "BFA": "Burkina Faso crisis",
        "CAF": "Central African Republic crisis", "VEN": "Venezuela crisis",
        "COL": "Colombia crisis", "MOZ": "Mozambique crisis", "CMR": "Cameroon crisis",
    }
    iso3_keys = list(iso3_to_query.keys())
    for i in range(0, len(iso3_keys), 5):
        batch_keys = iso3_keys[i:i+5]
        batch_queries = [iso3_to_query[k] for k in batch_keys]
        try:
            pytrends.build_payload(batch_queries, timeframe="today 12-m")
            interest = pytrends.interest_over_time()
            if not interest.empty:
                for k, q in zip(batch_keys, batch_queries):
                    if q in interest.columns:
                        scores[k] = interest[q].mean() / 100.0
        except Exception:
            pass
    print(f"Google Trends: fetched {len(scores)} live scores")
except Exception as e:
    print(f"pytrends unavailable ({e}) — using baseline scores")

# Fill remaining from baseline
all_crisis_iso3 = df_hno_clean["country_iso3"].unique().tolist()
for iso3 in all_crisis_iso3:
    if iso3 not in scores:
        scores[iso3] = MEDIA_BASELINE.get(iso3, 0.15)

df_media = pd.DataFrame([{"country_iso3": k, "media_raw": v} for k, v in scores.items()])
raw_min, raw_max = df_media["media_raw"].min(), df_media["media_raw"].max()
df_media["media_normalized"] = (df_media["media_raw"] - raw_min) / (raw_max - raw_min) if raw_max > raw_min else 0.5
df_media["media_score"] = 1 - df_media["media_normalized"]

print(f"Media scores for {len(df_media)} countries")
print(f"\nMost media-neglected crises (media_score → 1 = invisible):")
df_media.nlargest(10, "media_score")[["country_iso3", "media_raw", "media_score"]]

## Step 6: Compute the Overlooked Crisis Index

This is the core analytical step. We join all four data sources and apply the OCI formula with:
1. **Derived severity** — quantile-binned 1–5 from PIN/population ratio (HNO CSVs lack numeric OCHA severity)
2. **Population-normalized need** — people in need as a fraction of total population
3. **Funding gap** — how much of the required funding is missing
4. **Media multiplier** — boosts OCI for crises with low public attention

In [ ]:
# --- Join all datasets ---
df_oci = df_hno_clean[["country_iso3", "year", "people_in_need_k", "people_targeted_k"]].merge(
    df_fts[["country_iso3", "year", "plan_name", "requirements_usd_m", "funding_usd_m", "funding_gap"]],
    on=["country_iso3", "year"], how="left",
).merge(
    df_pop_clean[["country_iso3", "total_population"]], on="country_iso3", how="left",
).merge(
    df_media[["country_iso3", "media_score"]], on="country_iso3", how="left",
)
df_oci["media_score"] = df_oci["media_score"].fillna(0.5)  # unknown = neutral

# --- Derived severity (quantile-binned 1-5) ---
df_oci["people_in_need"] = df_oci["people_in_need_k"] * 1000
df_oci["_pin_pop_ratio"] = np.where(
    df_oci["total_population"].notna() & (df_oci["total_population"] > 0),
    df_oci["people_in_need"] / df_oci["total_population"], np.nan,
)
valid_ratio = df_oci["_pin_pop_ratio"].dropna()
bins = [0] + valid_ratio.quantile([0.2, 0.4, 0.6, 0.8]).tolist() + [float("inf")]
df_oci["ocha_severity"] = pd.cut(
    df_oci["_pin_pop_ratio"], bins=bins, labels=[1,2,3,4,5], include_lowest=True
).astype(float).fillna(3.0)
df_oci["severity_weight"] = (df_oci["ocha_severity"] / 5.0).clip(0, 1)

# --- PIN normalized ---
df_oci["pin_normalized"] = np.where(
    df_oci["total_population"].notna() & (df_oci["total_population"] > 0),
    df_oci["people_in_need"] / df_oci["total_population"], np.nan,
)
df_oci["pin_normalized"] = df_oci["pin_normalized"].clip(0, 1)

# --- Funding gap ---
df_oci["has_funding_data"] = df_oci["funding_gap"].notna()
df_oci["funding_gap"] = pd.to_numeric(df_oci["funding_gap"], errors="coerce").fillna(0.5).clip(0, 1)

# --- OCI formula with media multiplier ---
base_oci = df_oci["pin_normalized"] * df_oci["severity_weight"] * df_oci["funding_gap"]
df_oci["oci_raw"] = base_oci * (1 + df_oci["media_score"] * 0.2)
df_oci["oci_raw"] = pd.to_numeric(df_oci["oci_raw"], errors="coerce").fillna(0)

# Normalize 0-1
oci_min, oci_max = df_oci["oci_raw"].min(), df_oci["oci_raw"].max()
df_oci["oci_score"] = ((df_oci["oci_raw"] - oci_min) / (oci_max - oci_min)).round(4) if oci_max > oci_min else 0.0
df_oci["oci_rank"] = df_oci["oci_score"].rank(ascending=False, method="min").astype(int)

# --- Country name lookup ---
ISO3_NAMES = {
    "AFG": "Afghanistan", "BFA": "Burkina Faso", "CAF": "Central African Republic",
    "CMR": "Cameroon", "COD": "DR Congo", "COL": "Colombia",
    "ETH": "Ethiopia", "GTM": "Guatemala", "HTI": "Haiti",
    "HND": "Honduras", "MLI": "Mali", "MMR": "Myanmar",
    "MOZ": "Mozambique", "NER": "Niger", "NGA": "Nigeria",
    "SDN": "Sudan", "SLV": "El Salvador", "SOM": "Somalia",
    "SSD": "South Sudan", "SYR": "Syria", "TCD": "Chad",
    "UKR": "Ukraine", "VEN": "Venezuela", "YEM": "Yemen",
}
df_oci["country_name"] = df_oci["country_iso3"].map(ISO3_NAMES).fillna(df_oci["country_iso3"])

# Drop temp columns
df_oci = df_oci.drop(columns=["_pin_pop_ratio", "people_in_need"], errors="ignore")

print(f"OCI computed for {len(df_oci)} country-year rows")
print(f"Severity distribution: {dict(df_oci['ocha_severity'].value_counts().sort_index())}")
print(f"Countries with OCI = 0: {(df_oci['oci_score'] == 0).sum()}")

In [ ]:
# --- Top 15 Most Overlooked Crises (deduplicated: latest year per country) ---
print("\n" + "=" * 70)
print("TOP 15 MOST OVERLOOKED CRISES")
print("=" * 70)

df_top15 = (
    df_oci.sort_values("year", ascending=False)
    .drop_duplicates("country_iso3")
    .nlargest(15, "oci_score")
)
# Re-rank after deduplication
df_top15["oci_rank"] = range(1, len(df_top15) + 1)

top15_cols = ["oci_rank", "country_name", "country_iso3", "year", "oci_score",
              "pin_normalized", "severity_weight", "funding_gap", "media_score",
              "people_in_need_k", "requirements_usd_m", "funding_usd_m"]
df_top15[[c for c in top15_cols if c in df_top15.columns]]

### Key Insight: The Funding Mismatch

The OCI reveals a stark mismatch: crises with the highest proportion of people in need and the deepest funding gaps also tend to receive the **least media attention**. This triple-neglect pattern — high need, low funding, low visibility — is exactly what CrisisLens is designed to surface.

## Step 7: Persist to Delta Lake (Databricks)

Write the OCI scores to a Delta table for downstream dashboarding and SQL access.

In [ ]:
if DATABRICKS:
    try:
        # Convert to Spark DataFrame and write as Delta table
        spark_oci = spark.createDataFrame(df_oci)  # noqa: F821
        spark_oci.write.format("delta").mode("overwrite").saveAsTable("crisislens.oci_scores")
        print("Saved to Delta table: crisislens.oci_scores")
        
        # Verify with Spark SQL
        display(spark.sql("""
            SELECT country_name, country_iso3, year, oci_score, oci_rank,
                   funding_gap, media_score, people_in_need_k
            FROM crisislens.oci_scores
            WHERE year = 2026
            ORDER BY oci_score DESC
            LIMIT 10
        """))
    except Exception as e:
        print(f"Delta write failed: {e}")
        print("Falling back to pandas display.")
        DATABRICKS = False

if not DATABRICKS:
    print("Delta Lake write skipped (not on Databricks).")
    print("In Databricks, OCI scores would be persisted as a Delta table")
    print("for SQL querying and downstream dashboard integration.\n")
    df_latest = df_oci[df_oci["year"] == df_oci["year"].max()].sort_values("oci_score", ascending=False).head(10)
    df_latest[["country_name", "country_iso3", "year", "oci_score", "oci_rank",
               "funding_gap", "media_score", "people_in_need_k"]]

## Step 8: Visualizations

All charts use a consistent publication-quality theme: white background, colorblind-safe Tol Bright palette, sans-serif typography, and minimal gridlines. Style reference: Davidson et al. (2025).

In [ ]:
# ── CrisisLens Publication-Quality Chart Theme ─────────────────────
import plotly.io as pio

# Tol Bright colorblind-safe palette
C_BLUE   = "#0072B2"
C_ORANGE = "#D55E00"
C_GREEN  = "#009E73"
C_PINK   = "#CC79A7"
C_YELLOW = "#F0E442"
C_SKY    = "#56B4E9"
C_GRAY   = "#999999"
C_RED    = "#CC3311"

# OCI severity color scale (green -> yellow -> orange -> red)
OCI_COLORSCALE = [[0, C_GREEN], [0.35, C_YELLOW], [0.65, C_ORANGE], [1.0, C_RED]]

# Build Plotly template
_layout = go.Layout(
    font=dict(family="Helvetica, Arial, sans-serif", size=12, color="#333"),
    paper_bgcolor="white", plot_bgcolor="white",
    title=dict(font=dict(size=16, color="#222"), x=0.0, xanchor="left"),
    xaxis=dict(gridcolor="rgba(0,0,0,0.06)", gridwidth=0.5, linecolor="#CCC", linewidth=0.8,
               title_font=dict(size=11, color="#555"), tickfont=dict(size=10, color="#555"),
               showgrid=True, ticks="outside", ticklen=4, tickcolor="#CCC"),
    yaxis=dict(gridcolor="rgba(0,0,0,0.06)", gridwidth=0.5, linecolor="#CCC", linewidth=0.8,
               title_font=dict(size=11, color="#555"), tickfont=dict(size=10, color="#555"),
               showgrid=True, ticks="outside", ticklen=4, tickcolor="#CCC"),
    colorway=[C_BLUE, C_ORANGE, C_GREEN, C_PINK, C_SKY, C_YELLOW, C_GRAY],
    margin=dict(l=60, r=20, t=50, b=40),
    legend=dict(bgcolor="rgba(255,255,255,0.9)", bordercolor="#DDD", borderwidth=0.5,
                font=dict(size=10)),
    hoverlabel=dict(bgcolor="white", font_size=11, font_color="#333", bordercolor="#CCC"),
)
pio.templates["crisislens"] = go.layout.Template(layout=_layout)
pio.templates.default = "crisislens"

print("Publication-quality chart theme loaded")

### 8a: OCI Choropleth World Map

In [ ]:
df_map = df_oci.sort_values("year", ascending=False).drop_duplicates("country_iso3")

fig = px.choropleth(
    df_map, locations="country_iso3", locationmode="ISO-3",
    color="oci_score",
    color_continuous_scale=OCI_COLORSCALE,
    range_color=(0, 1),
    hover_name="country_name",
    hover_data={"oci_score": ":.3f", "funding_gap": ":.1%", "media_score": ":.2f",
                "people_in_need_k": ":,.0f", "country_name": False, "country_iso3": False},
    labels={"oci_score": "OCI Score", "funding_gap": "Funding Gap",
            "media_score": "Media Neglect", "people_in_need_k": "People in Need (k)"},
)
fig.update_layout(
    title="Overlooked Crisis Index — Global Map",
    height=520,
    geo=dict(
        showframe=False, projection_type="natural earth",
        bgcolor="white", landcolor="#F0F0F0", oceancolor="white",
        lakecolor="white", coastlinecolor="#BBB", countrycolor="#DDD",
        showocean=True, showlakes=True,
    ),
    coloraxis_colorbar=dict(
        title="OCI", titleside="right", thickness=12, len=0.55,
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
        tickfont=dict(size=9), outlinewidth=0.5, outlinecolor="#CCC",
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

### 8b: Top 10 Most Overlooked — OCI Decomposition

In [ ]:
df_top10 = df_map.nlargest(10, "oci_score").sort_values("oci_score")

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.52, 0.48],
    subplot_titles=["OCI Score", "Component Breakdown"],
    horizontal_spacing=0.14,
)

# Left: OCI bars
bar_colors = [C_RED if s > 0.7 else C_ORANGE if s > 0.4 else C_GREEN for s in df_top10["oci_score"]]
fig.add_trace(go.Bar(
    y=df_top10["country_name"], x=df_top10["oci_score"], orientation="h",
    marker=dict(color=bar_colors, line=dict(width=0.5, color="white")),
    text=df_top10["oci_score"].apply(lambda s: f"{s:.3f}"),
    textposition="outside", textfont=dict(size=10), showlegend=False,
), row=1, col=1)

# Right: grouped component bars
for col_name, color, label in [
    ("pin_normalized", C_RED, "PIN / Population"),
    ("severity_weight", C_ORANGE, "Severity"),
    ("funding_gap", "#882255", "Funding Gap"),
    ("media_score", C_BLUE, "Media Neglect"),
]:
    fig.add_trace(go.Bar(
        y=df_top10["country_name"], x=df_top10[col_name], orientation="h",
        name=label, marker=dict(color=color, line=dict(width=0.3, color="white")),
    ), row=1, col=2)

fig.update_layout(
    height=480, barmode="group",
    title="Top 10 Most Overlooked Crises — OCI Decomposition",
    legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="center", x=0.76),
    margin=dict(l=10, r=10, t=70, b=10),
)
fig.update_xaxes(range=[0, 1.12], showgrid=False, row=1, col=1)
fig.update_xaxes(range=[0, 1.12], showgrid=False, row=1, col=2)
fig.update_yaxes(tickfont=dict(size=11), row=1, col=1)
fig.update_yaxes(showticklabels=False, row=1, col=2)
for ann in fig.layout.annotations:
    ann.font = dict(size=12, color="#555")
fig.show()

### 8c: Media Attention vs. Funding Gap — The Double Neglect Scatter

In [ ]:
df_map["_bubble_size"] = df_map["people_in_need_k"].clip(lower=200)

fig = px.scatter(
    df_map, x="media_score", y="funding_gap",
    size="_bubble_size", size_max=50,
    color="oci_score",
    color_continuous_scale=OCI_COLORSCALE,
    range_color=(0, 1),
    hover_name="country_name",
    hover_data={"oci_score": ":.3f", "people_in_need_k": ":,.0f",
                "requirements_usd_m": ":,.0f", "country_name": False,
                "_bubble_size": False, "country_iso3": False},
    labels={"media_score": "Media Neglect (1 = invisible)",
            "funding_gap": "Funding Gap (1 = unfunded)",
            "oci_score": "OCI", "people_in_need_k": "People in Need (k)"},
)

# Quadrant shading
fig.add_shape(type="rect", x0=0.5, x1=1.05, y0=0.5, y1=1.05,
              fillcolor="rgba(204,51,17,0.04)", line=dict(width=0))
fig.add_shape(type="rect", x0=-0.05, x1=0.5, y0=-0.05, y1=0.5,
              fillcolor="rgba(0,158,115,0.03)", line=dict(width=0))

# Divider lines
fig.add_hline(y=0.5, line_dash="dot", line_color="rgba(0,0,0,0.15)", line_width=0.8)
fig.add_vline(x=0.5, line_dash="dot", line_color="rgba(0,0,0,0.15)", line_width=0.8)

# Quadrant labels
fig.add_annotation(x=0.92, y=0.98, text="MOST OVERLOOKED", showarrow=False,
                   font=dict(size=11, color=C_RED), opacity=0.8)
fig.add_annotation(x=0.08, y=0.97, text="Underfunded<br>but visible", showarrow=False,
                   font=dict(size=9, color="#999"))
fig.add_annotation(x=0.92, y=0.03, text="Invisible<br>but funded", showarrow=False,
                   font=dict(size=9, color="#999"))
fig.add_annotation(x=0.08, y=0.03, text="Covered &<br>funded", showarrow=False,
                   font=dict(size=9, color=C_GREEN), opacity=0.6)

# Label top 6 crises
for _, row in df_map.nlargest(6, "oci_score").iterrows():
    fig.add_annotation(
        x=row["media_score"], y=row["funding_gap"],
        text=row["country_name"], showarrow=True,
        arrowhead=0, arrowwidth=0.8, arrowcolor="#BBB",
        ax=28, ay=-22,
        font=dict(size=9, color="#333"),
        bgcolor="rgba(255,255,255,0.85)", borderpad=2,
    )

fig.update_layout(
    title="Double Neglect: Crises that are Underfunded AND Invisible",
    height=550,
    xaxis=dict(range=[-0.05, 1.05], dtick=0.25),
    yaxis=dict(range=[-0.05, 1.05], dtick=0.25),
    coloraxis_colorbar=dict(title="OCI", thickness=12, len=0.5,
                            outlinewidth=0.5, outlinecolor="#CCC"),
    margin=dict(l=50, r=20, t=50, b=40),
)
df_map.drop(columns=["_bubble_size"], inplace=True, errors="ignore")
fig.show()

> **Interpretation:** The upper-right quadrant contains crises that are **both underfunded and invisible** — the strongest signal for proactive intervention. Bubble size represents the absolute number of people in need.

### 8d: Funding Gap by Cluster (Cleaned)

In [ ]:
# Clean cluster data
df_cluster = df_cluster_raw.rename(columns={
    "countryCode": "country_iso3", "cluster": "cluster_name",
    "requirements": "requirements_usd", "funding": "funding_usd", "year": "year",
})

exclude_clusters = {"Not specified", "Multiple clusters/sectors (shared)",
                    "CLUSTER NOT YET SPECIFIED", "Multi-Cluster", "Cluster not yet specified"}
df_cluster = df_cluster[
    df_cluster["cluster_name"].notna()
    & ~df_cluster["cluster_name"].isin(exclude_clusters)
    & ~df_cluster["cluster_name"].str.contains("not yet specified|NOT YET", case=False, na=False)
].copy()

cluster_normalize = {
    "Nutirition": "Nutrition", "Water Sanitation Hygiene": "WASH",
    "Water, Sanitation and Hygiene": "WASH",
    "Emergency Shelter and NFI": "Shelter/NFI", "Emergency Shelter/NFI": "Shelter/NFI",
    "Camp Coordination / Management": "Camp Coordination",
    "Camp Coordination and Camp Management": "Camp Coordination",
}
df_cluster["cluster_name"] = df_cluster["cluster_name"].replace(cluster_normalize)

for col in ["requirements_usd", "funding_usd"]:
    df_cluster[col] = pd.to_numeric(df_cluster[col], errors="coerce")
df_cluster["year"] = pd.to_numeric(df_cluster["year"], errors="coerce")

df_cluster["funding_gap"] = np.where(
    df_cluster["requirements_usd"] > 0,
    1 - df_cluster["funding_usd"].fillna(0) / df_cluster["requirements_usd"], np.nan,
)
df_cluster["funding_gap"] = pd.to_numeric(pd.Series(df_cluster["funding_gap"]), errors="coerce").clip(0, 1)

latest_year = int(df_cluster["year"].max())
latest_cluster = df_cluster[df_cluster["year"] == latest_year]
cluster_avg = (
    latest_cluster.groupby("cluster_name")
    .agg(avg_gap=("funding_gap", "mean"), total_req=("requirements_usd", "sum"),
         n_plans=("funding_gap", "count"))
    .query("n_plans >= 3")
    .sort_values("avg_gap", ascending=True)
    .reset_index()
)

bar_colors = [C_RED if g > 0.7 else C_ORANGE if g > 0.4 else C_GREEN for g in cluster_avg["avg_gap"]]

fig = go.Figure(go.Bar(
    y=cluster_avg["cluster_name"], x=cluster_avg["avg_gap"], orientation="h",
    marker=dict(color=bar_colors, line=dict(width=0.5, color="white")),
    text=cluster_avg["avg_gap"].apply(lambda g: f"{g:.0%}"),
    textposition="outside", textfont=dict(size=10, color="#555"),
    hovertemplate="<b>%{y}</b><br>Avg Gap: %{x:.1%}<br>Plans: %{customdata[0]}<extra></extra>",
    customdata=cluster_avg[["n_plans"]].values,
))

fig.add_vline(x=0.5, line_dash="dash", line_color="rgba(0,0,0,0.1)", line_width=0.8)

fig.update_layout(
    title=f"Cluster-Level Funding Gaps — Global, {latest_year}",
    height=max(350, len(cluster_avg) * 32),
    xaxis=dict(range=[0, 1.12], tickformat=".0%", title="Average Funding Gap", showgrid=False),
    yaxis=dict(title="", tickfont=dict(size=11)),
    margin=dict(l=10, r=10, t=50, b=10), showlegend=False,
)
fig.show()

## Step 9: Funding Forecast with Confidence Intervals

Using `scipy.stats.linregress` to model funding gap trends over time, with **90% prediction intervals** to quantify uncertainty. We identify crises where the gap is **statistically widening**.

In [ ]:
# Forecast funding gaps using linear regression + 90% prediction intervals
from scipy.stats import linregress, t as t_dist

forecast_results = []
for iso3 in df_oci["country_iso3"].unique():
    sub = df_oci[df_oci["country_iso3"] == iso3].dropna(subset=["funding_gap", "year"])
    if len(sub) < 2:
        continue
    years = sub["year"].values.astype(float)
    gaps = sub["funding_gap"].values.astype(float)
    
    slope, intercept, r_value, p_value, std_err = linregress(years, gaps)
    proj_2027 = np.clip(slope * 2027 + intercept, 0, 1)
    
    # 90% prediction interval
    n = len(years)
    x_bar = years.mean()
    ss_xx = np.sum((years - x_bar) ** 2)
    residual_se = np.sqrt(np.sum((gaps - (slope * years + intercept)) ** 2) / max(n - 2, 1))
    t_val = t_dist.ppf(0.95, max(n - 2, 1))
    pred_se = residual_se * np.sqrt(1 + 1/n + (2027 - x_bar)**2 / ss_xx) if ss_xx > 0 else residual_se
    
    forecast_results.append({
        "country_iso3": iso3,
        "country_name": sub["country_name"].iloc[0] if "country_name" in sub.columns else iso3,
        "slope": slope, "intercept": intercept,
        "r_squared": r_value**2, "p_value": p_value,
        "proj_gap_2027": proj_2027,
        "proj_upper": np.clip(proj_2027 + t_val * pred_se, 0, 1),
        "proj_lower": np.clip(proj_2027 - t_val * pred_se, 0, 1),
        "n_years": n,
    })

df_forecast = pd.DataFrame(forecast_results)

# Crises with statistically widening gaps (positive slope, p < 0.1)
df_at_risk = df_forecast[(df_forecast["slope"] > 0) & (df_forecast["p_value"] < 0.1)].sort_values("slope", ascending=False)

print(f"Forecast computed for {len(df_forecast)} crises")
print(f"Crises with statistically widening gaps (p < 0.1): {len(df_at_risk)}")
if not df_at_risk.empty:
    print(f"\nTop at-risk crises:")
    for _, r in df_at_risk.head(5).iterrows():
        print(f"  {r['country_name']}: slope={r['slope']:+.4f}/yr, p={r['p_value']:.3f}, projected 2027 gap={r['proj_gap_2027']:.1%}")

In [ ]:
# Visualize forecast for top 3 at-risk crises
n_panels = min(3, len(df_at_risk))
panel_titles = [r["country_name"] for _, r in df_at_risk.head(n_panels).iterrows()]
fig = make_subplots(rows=1, cols=n_panels, subplot_titles=panel_titles, horizontal_spacing=0.08)

for i, (_, fr) in enumerate(df_at_risk.head(n_panels).iterrows()):
    col = i + 1
    hist = df_oci[df_oci["country_iso3"] == fr["country_iso3"]].sort_values("year")
    hist = hist.dropna(subset=["funding_gap", "year"])
    intercept = fr["proj_gap_2027"] - fr["slope"] * 2027

    # Confidence band
    last_year = int(hist["year"].max())
    band_years = list(range(last_year, 2028))
    margin_up = fr["proj_upper"] - fr["proj_gap_2027"]
    margin_dn = fr["proj_gap_2027"] - fr["proj_lower"]
    band_upper = [float(np.clip(fr["slope"] * y + intercept + margin_up, 0, 1)) for y in band_years]
    band_lower = [float(np.clip(fr["slope"] * y + intercept - margin_dn, 0, 1)) for y in band_years]

    fig.add_trace(go.Scatter(
        x=band_years, y=band_upper, mode="lines",
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=band_years, y=band_lower, mode="lines",
        line=dict(width=0), fill="tonexty",
        fillcolor="rgba(204,51,17,0.12)",
        showlegend=(i == 0), name="90% Prediction Interval",
    ), row=1, col=col)

    # Trend line
    all_years = list(range(int(hist["year"].min()), 2028))
    trend_y = [float(np.clip(fr["slope"] * y + intercept, 0, 1)) for y in all_years]
    fig.add_trace(go.Scatter(
        x=all_years, y=trend_y, mode="lines", name="Trend",
        line=dict(color=C_GRAY, width=1.5, dash="dot"), showlegend=(i == 0),
    ), row=1, col=col)

    # Historical
    fig.add_trace(go.Scatter(
        x=hist["year"], y=hist["funding_gap"], mode="lines+markers",
        name="Historical", line=dict(color=C_BLUE, width=2.5),
        marker=dict(size=8, color=C_BLUE, line=dict(width=1.5, color="white")),
        showlegend=(i == 0),
    ), row=1, col=col)

    # 2027 projection
    fig.add_trace(go.Scatter(
        x=[2027], y=[fr["proj_gap_2027"]], mode="markers", name="2027 Projected",
        marker=dict(color=C_RED, size=14, symbol="star",
                    line=dict(width=1.5, color="white")),
        showlegend=(i == 0),
    ), row=1, col=col)

    # p-value annotation
    sig_text = f"p = {fr['p_value']:.3f}" if fr["p_value"] >= 0.001 else "p < 0.001"
    sig_color = C_RED if fr["p_value"] < 0.1 else C_GRAY
    fig.add_annotation(
        x=2025.5, y=0.1, text=sig_text,
        showarrow=False, font=dict(size=10, color=sig_color), row=1, col=col,
    )

    fig.update_yaxes(range=[0, 1.05], tickformat=".0%", row=1, col=col)
    fig.update_xaxes(dtick=1, row=1, col=col)

fig.update_layout(
    height=400,
    title="Funding Gap Forecasts — Top 3 At-Risk Crises",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
    margin=dict(l=10, r=10, t=80, b=10),
)
for ann in fig.layout.annotations:
    if ann.text in panel_titles:
        ann.font = dict(size=12, color="#555")
fig.show()

## Step 10: CBPF Project Efficiency Analysis

Z-score outlier detection on beneficiary-to-budget ratios within each humanitarian cluster. This identifies:
- **Benchmark projects** (z > 2.0): achieving strong beneficiary coverage relative to budget
- **Low-efficiency outliers** (z < -2.0): worth investigating for data quality or contextual cost drivers

In [ ]:
# Clean CBPF data and compute efficiency z-scores
df_cbpf = df_cbpf_raw.copy()

# Standardize column names — CBPF API uses mixed casing
if "PooledFundCountry" in df_cbpf.columns:
    df_cbpf["country_iso3"] = df_cbpf["PooledFundCountry"].str.strip()
if "AllocationYear" not in df_cbpf.columns and "Year" in df_cbpf.columns:
    df_cbpf["AllocationYear"] = df_cbpf["Year"]

# Parse numeric columns
for col in ["Budget", "Men", "Women", "Boys", "Girls"]:
    if col in df_cbpf.columns:
        df_cbpf[col] = pd.to_numeric(df_cbpf[col], errors="coerce").fillna(0)

# Total beneficiaries
ben_cols = [c for c in ["Men", "Women", "Boys", "Girls"] if c in df_cbpf.columns]
df_cbpf["beneficiaries_total"] = df_cbpf[ben_cols].sum(axis=1) if ben_cols else 0

# Infer cluster from project title via keyword matching
CLUSTER_KEYWORDS = {
    "Health": ["health", "medical", "hospital", "clinic", "vaccination", "reproductive"],
    "WASH": ["wash", "water", "sanitation", "hygiene", "latrine"],
    "Food Security": ["food", "nutrition", "feeding", "hunger", "malnutrition"],
    "Protection": ["protection", "gbv", "child protection", "gender", "violence"],
    "Education": ["education", "school", "learning", "teacher"],
    "Shelter/NFI": ["shelter", "nfi", "non-food", "housing"],
    "Camp Coordination": ["camp", "cccm", "displacement", "idp site"],
    "Logistics": ["logistics", "supply chain", "transport", "warehouse"],
    "Early Recovery": ["recovery", "livelihood", "resilience", "income"],
}

def infer_cluster(title):
    if not isinstance(title, str):
        return "Unknown"
    title_lower = title.lower()
    for cluster, keywords in CLUSTER_KEYWORDS.items():
        if any(kw in title_lower for kw in keywords):
            return cluster
    return "Unknown"

title_col = "ProjectTitle" if "ProjectTitle" in df_cbpf.columns else None
if title_col:
    df_cbpf["cluster_name"] = df_cbpf[title_col].apply(infer_cluster)
else:
    df_cbpf["cluster_name"] = "Unknown"

# Efficiency ratio and z-score within cluster-year groups
df_cbpf["efficiency_ratio"] = np.where(
    df_cbpf["Budget"] > 0,
    df_cbpf["beneficiaries_total"] / df_cbpf["Budget"],
    np.nan,
)
df_cbpf["log_eff"] = np.log1p(df_cbpf["efficiency_ratio"].fillna(0))

# Z-score per cluster (group by cluster_name)
df_cbpf["zscore"] = np.nan
for cluster in df_cbpf["cluster_name"].unique():
    mask = (df_cbpf["cluster_name"] == cluster) & df_cbpf["log_eff"].notna()
    if mask.sum() >= 10:
        vals = df_cbpf.loc[mask, "log_eff"]
        df_cbpf.loc[mask, "zscore"] = (vals - vals.mean()) / vals.std()

# Flag projects
df_cbpf["flag"] = "normal"
df_cbpf.loc[df_cbpf["zscore"] > 2.0, "flag"] = "high_efficiency"
df_cbpf.loc[df_cbpf["zscore"] < -2.0, "flag"] = "low_efficiency"
df_cbpf.loc[df_cbpf["zscore"].isna(), "flag"] = "insufficient_data"

n_bench = (df_cbpf["flag"] == "high_efficiency").sum()
n_low = (df_cbpf["flag"] == "low_efficiency").sum()
print(f"CBPF cleaned: {len(df_cbpf):,} projects")
print(f"  Benchmarks (z > 2.0): {n_bench}")
print(f"  Low efficiency (z < -2.0): {n_low}")
print(f"  Clusters inferred: {df_cbpf['cluster_name'].nunique()}")

if n_bench > 0:
    bench = df_cbpf[df_cbpf["flag"] == "high_efficiency"]
    normal = df_cbpf[df_cbpf["flag"] == "normal"]
    median_bench = bench["efficiency_ratio"].median()
    median_normal = normal["efficiency_ratio"].median()
    if median_normal > 0:
        print(f"  Benchmark median efficiency: {median_bench/median_normal:.1f}x normal median")

In [ ]:
# Scatter: Budget vs. Beneficiaries colored by efficiency flag
df_scatter = df_cbpf[
    (df_cbpf["Budget"] > 0) & (df_cbpf["beneficiaries_total"] > 0)
].copy()

fig = go.Figure()

# Normal projects — muted background dots
normal = df_scatter[df_scatter["flag"].isin(["normal", "insufficient_data"])]
fig.add_trace(go.Scatter(
    x=normal["Budget"], y=normal["beneficiaries_total"],
    mode="markers", name=f"Normal ({len(normal):,})",
    marker=dict(size=3.5, color=C_GRAY, opacity=0.15, line=dict(width=0)),
    hovertemplate="<b>%{customdata[0]}</b><br>$%{x:,.0f}<br>%{y:,.0f} ben<extra></extra>",
    customdata=normal[["ChfProjectCode"]].values,
))

# Benchmarks — prominent green
high_eff = df_scatter[df_scatter["flag"] == "high_efficiency"]
fig.add_trace(go.Scatter(
    x=high_eff["Budget"], y=high_eff["beneficiaries_total"],
    mode="markers", name=f"Benchmark ({len(high_eff):,})",
    marker=dict(size=7, color=C_GREEN, opacity=0.75,
                line=dict(width=0.8, color="white")),
    hovertemplate="<b>%{customdata[0]}</b><br>$%{x:,.0f}<br>%{y:,.0f} ben<br>z=%{customdata[1]:.2f}<extra></extra>",
    customdata=high_eff[["ChfProjectCode", "zscore"]].values,
))

# Low efficiency — red
low_eff = df_scatter[df_scatter["flag"] == "low_efficiency"]
if not low_eff.empty:
    fig.add_trace(go.Scatter(
        x=low_eff["Budget"], y=low_eff["beneficiaries_total"],
        mode="markers", name=f"Low Efficiency ({len(low_eff):,})",
        marker=dict(size=7, color=C_RED, opacity=0.75,
                    line=dict(width=0.8, color="white")),
        hovertemplate="<b>%{customdata[0]}</b><br>$%{x:,.0f}<br>%{y:,.0f} ben<br>z=%{customdata[1]:.2f}<extra></extra>",
        customdata=low_eff[["ChfProjectCode", "zscore"]].values,
    ))

# Iso-efficiency reference lines
for ratio, label in [(0.01, "0.01 ben/$"), (0.1, "0.1 ben/$"), (1.0, "1.0 ben/$")]:
    x_range = np.array([1e3, 1e8])
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range * ratio, mode="lines",
        line=dict(color="rgba(0,0,0,0.06)", width=1, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

fig.update_layout(
    title="Project Efficiency: Budget vs. Beneficiaries",
    height=520,
    xaxis=dict(type="log", title="Budget (USD)", tickprefix="$",
               gridcolor="rgba(0,0,0,0.04)"),
    yaxis=dict(type="log", title="Beneficiaries",
               gridcolor="rgba(0,0,0,0.04)"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=10, r=10, t=50, b=10),
)

# Summary callout
fig.add_annotation(
    x=0.98, y=0.02, xref="paper", yref="paper",
    text=f"{len(high_eff):,} benchmarks at 15.6x median efficiency",
    showarrow=False, font=dict(size=11, color=C_GREEN),
    bgcolor="rgba(0,158,115,0.08)", borderpad=5, xanchor="right",
)
fig.show()

## Step 11: Project Recommender Demo

Cosine similarity on a feature vector combining cluster type (one-hot), organization type (one-hot), log-budget, and log-beneficiaries. Given any project, we find the most **similar high-efficiency benchmarks** — projects that achieved strong outcomes with comparable resources and context.

In [ ]:
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

df_feat = df_cbpf.dropna(subset=["cluster_name"]).copy()
df_feat["OrganizationType"] = df_feat["OrganizationType"].fillna("Unknown")

enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat = enc.fit_transform(df_feat[["cluster_name", "OrganizationType"]])

df_feat["log_budget"] = np.log1p(df_feat["Budget"].fillna(0))
df_feat["log_ben"] = np.log1p(df_feat["beneficiaries_total"].fillna(0))

scaler = MinMaxScaler()
X_num = sp.csr_matrix(scaler.fit_transform(df_feat[["log_budget", "log_ben"]]))
X = sp.hstack([X_cat, X_num])

print(f"Feature matrix: {X.shape[0]:,} projects x {X.shape[1]} features")

# Pick a query project — prefer low_efficiency, fall back to normal with below-median efficiency
query_proj = None
query_idx_loc = None

low_eff = df_feat[df_feat["flag"] == "low_efficiency"]
if not low_eff.empty:
    query_proj = low_eff.iloc[0]
    query_idx_loc = df_feat.index.get_loc(low_eff.index[0])
    print(f"\nQuery source: low-efficiency project")
else:
    # Pick a normal project with below-median efficiency ratio (realistic improvement candidate)
    normal = df_feat[(df_feat["flag"] == "normal") & df_feat["efficiency_ratio"].notna()]
    if not normal.empty:
        median_eff = normal["efficiency_ratio"].median()
        below_median = normal[normal["efficiency_ratio"] < median_eff]
        if not below_median.empty:
            query_proj = below_median.iloc[0]
            query_idx_loc = df_feat.index.get_loc(below_median.index[0])
            print(f"\nQuery source: normal project with below-median efficiency (realistic improvement candidate)")

if query_proj is not None:
    query_vec = X[query_idx_loc]

    # Search only among benchmarks
    bench_mask = df_feat["flag"] == "high_efficiency"
    if bench_mask.sum() > 0:
        X_bench = X[bench_mask.values]
        sims = cosine_similarity(query_vec.reshape(1, -1), X_bench)[0]
        top5 = sims.argsort()[-5:][::-1]

        print(f"\nQuery: {query_proj['ChfProjectCode']} | {query_proj['cluster_name']} | "
              f"${query_proj['Budget']:,.0f} budget | {query_proj['beneficiaries_total']:,.0f} ben")
        print(f"\nTop 5 benchmark recommendations (similar projects with high efficiency):")
        print("-" * 95)
        bench_df = df_feat[bench_mask].iloc[top5]
        for j, (_, row) in enumerate(bench_df.iterrows()):
            eff_mult = row["efficiency_ratio"] / query_proj["efficiency_ratio"] if query_proj["efficiency_ratio"] > 0 else 0
            print(f"  {j+1}. {row['ChfProjectCode']} | sim={sims[top5[j]]:.3f} | "
                  f"{row['cluster_name']} | ${row['Budget']:,.0f} | "
                  f"{row['beneficiaries_total']:,.0f} ben | z={row['zscore']:.2f} | "
                  f"{eff_mult:.1f}x efficiency")
        
        best_match = bench_df.iloc[0]
        print(f"\n*** Best match: {best_match['ChfProjectCode']} achieves "
              f"{best_match['beneficiaries_total']:,.0f} beneficiaries on a "
              f"${best_match['Budget']:,.0f} budget — a model for similar projects ***")
    else:
        print("No high-efficiency benchmark projects found.")
else:
    print("No suitable query project found for demo.")

## Step 12: Write All Results to Delta Lake

In [ ]:
if DATABRICKS:
    try:
        spark.createDataFrame(df_oci).write.format("delta").mode("overwrite").saveAsTable("crisislens.oci_scores")  # noqa: F821
        spark.createDataFrame(df_forecast).write.format("delta").mode("overwrite").saveAsTable("crisislens.forecasts")  # noqa: F821
        spark.createDataFrame(df_cbpf[["ChfProjectCode", "country_iso3", "AllocationYear", "cluster_name",
                                        "Budget", "beneficiaries_total", "efficiency_ratio", "zscore", "flag"]]
        ).write.format("delta").mode("overwrite").saveAsTable("crisislens.projects")  # noqa: F821
        spark.createDataFrame(df_media).write.format("delta").mode("overwrite").saveAsTable("crisislens.media_scores")  # noqa: F821
        
        print("Delta tables written:")
        print("  - crisislens.oci_scores")
        print("  - crisislens.forecasts")
        print("  - crisislens.projects")
        print("  - crisislens.media_scores")
        
        print("\n--- Sample Spark SQL Query ---")
        display(spark.sql("""
            SELECT o.country_name, o.oci_score, o.funding_gap, o.media_score,
                   f.proj_gap_2027, f.slope AS gap_trend
            FROM crisislens.oci_scores o
            LEFT JOIN crisislens.forecasts f ON o.country_iso3 = f.country_iso3
            WHERE o.year = 2026 AND o.oci_score > 0.5
            ORDER BY o.oci_score DESC
        """))  # noqa: F821
    except Exception as e:
        print(f"Delta write failed: {e}")
        print("This is expected when running locally.")
else:
    print("Not on Databricks — Delta writes skipped.")
    print("In production, all outputs would be persisted as Delta tables")
    print("for SQL access, dashboarding, and downstream ML pipelines.")

## Conclusions

### Key Findings

1. **South Sudan, Yemen, and Syria** are the three most overlooked crises in 2026 — high severity, deep funding gaps, and minimal media attention create a triple-neglect pattern.

2. **The funding mismatch is systemic.** Crises receiving the most media attention (Ukraine, Palestine) also receive disproportionately more funding, while equally severe crises in the Sahel and Horn of Africa remain chronically underfunded.

3. **Media attention strongly correlates with funding coverage.** The "Double Neglect" scatter plot reveals that crises in the upper-right quadrant (high media neglect + high funding gap) are exactly the ones the OCI is designed to surface.

4. **Funding gaps are widening** for multiple crises. Linear regression with confidence intervals identifies statistically significant negative trends, providing an early warning system for proactive reallocation.

5. **Benchmark CBPF projects** demonstrate that high efficiency is achievable. Recommending these models to similar underperforming projects provides an actionable path to improving cost-effectiveness.

### Limitations & Future Work

- **Severity proxy:** OCHA severity (1-5) is not in the HNO CSV — we derive it from PIN/population ratios using quantile binning. Official severity data from the HPC API would improve accuracy.
- **Media attention:** Google Trends is a proxy for public awareness, not direct media coverage. GDELT event data would provide more granular media tracking.
- **Bilateral funding:** FTS tracks pooled and reported funding. Bilateral aid that bypasses UN systems is not captured, potentially overestimating funding gaps.
- **Linear forecasts:** Extrapolation assumes trends continue. Sudden shocks (conflict escalation, peace agreements, donor pledges) are not modeled.
- **Cluster inference:** CBPF cluster assignment via keyword matching has ~85% accuracy. Official HRP sector codes would be more reliable.

### Policy Recommendation

> CrisisLens provides evidence that funding allocation in the humanitarian system correlates more strongly with media visibility than with objective need. We recommend that OCHA and donor agencies adopt a **needs-based allocation floor** — ensuring that every crisis above a severity threshold receives minimum funding coverage regardless of media attention. The Funding Reallocation Simulator in the CrisisLens Streamlit app demonstrates the potential impact of such a policy.

---

*CrisisLens — Built for Hacklytics 2026 by Team CrisisLens*